In [12]:
from pathlib import Path
import re
import pandas as pd
import pymupdf
from openai import OpenAI

In [13]:
client = OpenAI()

In [11]:
schema_veiculo = {
    "type": "object",
    "properties": {
        "identificacao": {
            "type": "object",
            "properties": {
                "nome_veiculo": {
                    "type": "string",
                    "description": "Nome completo do veículo conforme aparece no PDF."
                },
                "marca": {
                    "type": ["string", "null"],
                    "description": "Marca do veículo, quando identificável."
                },
                "modelo": {
                    "type": ["string", "null"],
                    "description": "Modelo do veículo, quando identificável."
                },
                "versao": {
                    "type": ["string", "null"],
                    "description": "Versão ou acabamento do veículo, quando identificável."
                },
                "ano": {
                    "type": ["integer", "null"],
                    "description": "Ano do veículo."
                },
                "preco": {
                    "type": ["number", "null"],
                    "description": "Preço em reais, convertido para número. Exemplo: R$ 249.990 deve virar 249990."
                }
            },
            "required": [
                "nome_veiculo",
                "marca",
                "modelo",
                "versao",
                "ano",
                "preco"
            ],
            "additionalProperties": False
        },

        "ficha_tecnica": {
            "type": "array",
            "description": "Lista de seções da ficha técnica. Cada seção contém atributos no formato nome-valor.",
            "items": {
                "type": "object",
                "properties": {
                    "secao": {
                        "type": "string",
                        "description": "Nome da seção da ficha técnica. Exemplos: Geral, Motor a combustão, Motor elétrico, Transmissão, Suspensão, Freios, Direção, Pneus, Dimensões, Desempenho, Consumo, Autonomia."
                    },
                    "atributos": {
                        "type": "array",
                        "description": "Lista de atributos técnicos encontrados dentro da seção.",
                        "items": {
                            "type": "object",
                            "properties": {
                                "nome": {
                                    "type": "string",
                                    "description": "Nome do atributo técnico exatamente como aparece ou de forma padronizada. Exemplo: Propulsão, Combustível, Potência máxima, Torque máximo."
                                },
                                "valor": {
                                    "type": ["string", "number", "integer", "null"],
                                    "description": "Valor principal do atributo. Pode ser texto ou número."
                                },
                                "unidade": {
                                    "type": ["string", "null"],
                                    "description": "Unidade de medida, quando houver. Exemplos: cv, kgfm, km/l, mm, litros, kg, km, kWh."
                                },
                                "valor_original": {
                                    "type": ["string", "null"],
                                    "description": "Texto original do valor como aparece no PDF, preservando unidade e observações."
                                },
                                "observacao": {
                                    "type": ["string", "null"],
                                    "description": "Observação adicional, complemento ou nota associada ao atributo, quando houver."
                                }
                            },
                            "required": [
                                "nome",
                                "valor",
                                "unidade",
                                "valor_original",
                                "observacao"
                            ],
                            "additionalProperties": False
                        }
                    }
                },
                "required": [
                    "secao",
                    "atributos"
                ],
                "additionalProperties": False
            }
        },

        "equipamentos": {
            "type": "array",
            "description": "Lista de categorias de equipamentos presentes no veículo.",
            "items": {
                "type": "object",
                "properties": {
                    "categoria": {
                        "type": "string",
                        "description": "Categoria do equipamento. Exemplos: Segurança, Conforto, Infotenimento."
                    },
                    "itens": {
                        "type": "array",
                        "description": "Lista de equipamentos da categoria.",
                        "items": {
                            "type": "object",
                            "properties": {
                                "nome": {
                                    "type": "string",
                                    "description": "Nome do equipamento."
                                },
                                "disponibilidade": {
                                    "type": "string",
                                    "enum": [
                                        "serie",
                                        "opcional",
                                        "nao_informado"
                                    ],
                                    "description": "Indica se o equipamento é de série, opcional ou se a disponibilidade não foi identificada."
                                },
                                "valor_original": {
                                    "type": ["string", "null"],
                                    "description": "Texto original do equipamento como aparece no PDF."
                                }
                            },
                            "required": [
                                "nome",
                                "disponibilidade",
                                "valor_original"
                            ],
                            "additionalProperties": False
                        }
                    }
                },
                "required": [
                    "categoria",
                    "itens"
                ],
                "additionalProperties": False
            }
        },

        "metadados_extracao": {
            "type": "object",
            "properties": {
                "fonte": {
                    "type": ["string", "null"],
                    "description": "Fonte do documento, quando identificável."
                },
                "paginas_utilizadas": {
                    "type": "array",
                    "items": {
                        "type": "integer"
                    },
                    "description": "Páginas usadas na extração."
                },
                "campos_incertos": {
                    "type": "array",
                    "items": {
                        "type": "string"
                    },
                    "description": "Campos ou trechos cuja extração parece incerta."
                },
                "observacoes": {
                    "type": ["string", "null"],
                    "description": "Observações gerais sobre problemas de leitura, inconsistências ou limitações da extração."
                }
            },
            "required": [
                "fonte",
                "paginas_utilizadas",
                "campos_incertos",
                "observacoes"
            ],
            "additionalProperties": False
        }
    },
    "required": [
        "identificacao",
        "ficha_tecnica",
        "equipamentos",
        "metadados_extracao"
    ],
    "additionalProperties": False
}

In [14]:
base_dir = Path('carros_benchmark')
pdf_paths = sorted(base_dir.glob('*.pdf'))
pdf_paths

[WindowsPath('carros_benchmark/Carros na Web _ BYD Song Plus GS 1.5 2027 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf'),
 WindowsPath('carros_benchmark/Carros na Web _ BYD Song Plus Premium 1.5 AWD 2027 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf'),
 WindowsPath('carros_benchmark/Carros na Web _ Jeep Compass Night Eagle 1.3 2026 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf'),
 WindowsPath('carros_benchmark/Carros na Web _ Jeep Compass Sport 1.3 2026 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf'),
 WindowsPath('carros_benchmark/Carros na Web _ Renault Boreal Evolution 1.3 2026 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf'),
 WindowsPath('carros_benchmark/Carros na Web _ Renault Boreal Iconic 1.3 2026 _ Ficha Técnica, Especificações, Equipamentos, Fotos, Preço_.pdf'),
 WindowsPath('carros_benchmark/Carros na Web _ Renault Boreal Techno 1.3 2026 _ Ficha Técnica, Especificações, Equipamento

In [20]:
import re
from pathlib import Path
import fitz  # PyMuPDF


def limpar_linha(linha: str) -> str:
    linha = linha.replace("\xa0", " ")
    linha = re.sub(r"\s+", " ", linha)
    return linha.strip()


def linha_eh_ruido(linha: str) -> bool:
    if not linha:
        return True

    padroes_ruido = [
        r"^Página Principal",
        r"^Catálogo",
        r"^Comparativo",
        r"^Avaliação",
        r"^Ranking",
        r"^Opinião do Dono",
        r"^Notícias",
        r"^Carros Mais Vendidos",
        r"^Compartilhe",
        r"^Busca detalhada",
        r"^Post$",
        r"^Fotos$",
        r"^Legenda:",
        r"^As informações no website",
        r"^Algumas informações",
        r"^Material ilustrativo",
        r"^Sobre as informações",
        r"^Direitos Autorais",
        r"^Valor aproximado",
        r"^Preço médio aproximado",
        r"^Antes de adquirir",
        r"^Mapa do site",
        r"^Privacidade",
        r"^Termos de uso",
        r"^Mobile",
        r"^Fale Conosco",
        r"^\d+[,\.]\d+\s*s$",
    ]

    return any(
        re.search(padrao, linha, flags=re.IGNORECASE)
        for padrao in padroes_ruido
    )


def ler_pdf_para_texto_llm(caminho_pdf: str | Path) -> str:
    """
    Lê todo o PDF e devolve um único texto contínuo,
    sem enumeração de páginas e sem marcadores artificiais.

    Ideal para enviar o texto limpo ao LLM.
    """
    caminho_pdf = Path(caminho_pdf)

    if not caminho_pdf.exists():
        raise FileNotFoundError(f"PDF não encontrado: {caminho_pdf}")

    linhas_extraidas = []

    with fitz.open(caminho_pdf) as doc:
        for page in doc:
            texto = page.get_text("text")

            for linha in texto.splitlines():
                linha = limpar_linha(linha)

                if linha_eh_ruido(linha):
                    continue

                linhas_extraidas.append(linha)

    texto_final = "\n".join(linhas_extraidas)

    # Remove excesso de quebras de linha
    texto_final = re.sub(r"\n{3,}", "\n\n", texto_final)

    return texto_final.strip()

def separar_blocos_carros_na_web(texto: str) -> dict:
    texto = texto.strip()
    texto = re.sub(r"\n{3,}", "\n\n", texto)

    marcador_equipamentos = re.search(
        r"\nEquipamentos\n",
        texto,
        flags=re.IGNORECASE
    )

    if marcador_equipamentos:
        texto_antes_equip = texto[:marcador_equipamentos.start()].strip()
        texto_equipamentos = texto[marcador_equipamentos.end():].strip()
    else:
        texto_antes_equip = texto
        texto_equipamentos = ""

    secoes_tecnicas = [
        "MOTOR A COMBUSTÃO",
        "MOTOR ELÉTRICO",
        "TRANSMISSÃO",
        "SUSPENSÃO",
        "FREIOS",
        "DIREÇÃO",
        "PNEUS",
        "DIMENSÕES",
        "DESEMPENHO",
        "CONSUMO",
        "AUTONOMIA",
    ]

    padrao_primeira_secao = (
        r"\n(" + "|".join(map(re.escape, secoes_tecnicas)) + r")\n"
    )

    match_primeira_secao = re.search(
        padrao_primeira_secao,
        texto_antes_equip,
        flags=re.IGNORECASE
    )

    if match_primeira_secao:
        identificacao = texto_antes_equip[:match_primeira_secao.start()].strip()
        ficha_tecnica = texto_antes_equip[match_primeira_secao.start():].strip()
    else:
        identificacao = ""
        ficha_tecnica = texto_antes_equip

    return {
        "identificacao": identificacao,
        "ficha_tecnica": ficha_tecnica,
        "equipamentos": texto_equipamentos,
    }


def montar_texto_final_para_llm(blocos: dict) -> str:
    return f"""
[IDENTIFICACAO]
{blocos["identificacao"]}

[FICHA_TECNICA]
{blocos["ficha_tecnica"]}

[EQUIPAMENTOS]
{blocos["equipamentos"]}
""".strip()

In [21]:
texto_limpo = ler_pdf_para_texto_llm(pdf_paths[0])

blocos = separar_blocos_carros_na_web(texto_limpo)

texto_final_llm = montar_texto_final_para_llm(blocos)

print(texto_final_llm[:3000])

[IDENTIFICACAO]
Ficha Técnica Busca detalhada
BYD Song Plus GS 1.5
Ano 2027
Preço R$ 249.990
Propulsão
Híbrido plug-in
Combustível
Eletricidade/gasolina
IPVA R$ 10.0001
Seguro R$ 6.7502
Revisões R$ 6.383 até 60.000 km
Procedência Importado
Garantia 6 anos
Configuração SUV
Porte Médio
Lugares 5
Portas 4
Série Ocean
Nota do leitor
8,3 Avalie
Índice CNW
10.638,22
Euro NCAP
5
277
Proteção adulto
90%
Proteção a pedestres
83%
Proteção infantil
86%
Assistência segurança
77%

[FICHA_TECNICA]
MOTOR A COMBUSTÃO
Instalação Dianteiro
Aspiração Turbocompressor
Disposição Transversal
Alimentação Injeção multiponto
Cilindros 4 em linha
Comando de válvulas Duplo no cabeçote
Tuchos Hidráulicos
Variação do comando Admissão
Acionamento comando Corrente
Válvulas por cilindro 4
Deslocamento
cm3
Potência máxima
cv a 6000 rpm
Torque máximo 22,4 kgfm a 4500 rpm
Peso/potência 13,77 kg/cv
Torque específico 15,0 kgfm/litro
Peso/torque 79,9 kg/kgfm
Potência específica 86,8 cv/litro
Viscosidade do óleo 0W-203
MOTO

In [22]:
response = client.responses.create(
    model="gpt-5.4-mini",
    input=[
        {
            "role": "system",
            "content": (
                "Extraia dados estruturados do PDF de ficha técnica de veículo. "
                "Na seção Ficha Técnica, represente os dados como seções contendo pares nome-valor. "
                "Na seção Equipamentos, represente cada categoria com sua lista de itens. "
                "Use apenas informações presentes no documento. "
                "Não invente campos ausentes."
            )
        },
        {
            "role": "user",
            "content": texto_final_llm
        }
    ],
    text={
        "format": {
            "type": "json_schema",
            "name": "extracao_veiculo",
            "schema": schema_veiculo,
            "strict": True
        }
    }
)

In [25]:
import json
dados_extraidos = json.loads(response.output_text)

In [26]:
dados_extraidos

{'identificacao': {'nome_veiculo': 'BYD Song Plus GS 1.5',
  'marca': 'BYD',
  'modelo': 'Song Plus',
  'versao': 'GS 1.5',
  'ano': 2027,
  'preco': 249990},
 'ficha_tecnica': [{'secao': 'Geral',
   'atributos': [{'nome': 'Propulsão',
     'valor': 'Híbrido plug-in',
     'unidade': None,
     'valor_original': 'Híbrido plug-in',
     'observacao': None},
    {'nome': 'Combustível',
     'valor': 'Eletricidade/gasolina',
     'unidade': None,
     'valor_original': 'Eletricidade/gasolina',
     'observacao': None},
    {'nome': 'Procedência',
     'valor': 'Importado',
     'unidade': None,
     'valor_original': 'Importado',
     'observacao': None},
    {'nome': 'Garantia',
     'valor': '6 anos',
     'unidade': None,
     'valor_original': '6 anos',
     'observacao': None},
    {'nome': 'Configuração',
     'valor': 'SUV',
     'unidade': None,
     'valor_original': 'SUV',
     'observacao': None},
    {'nome': 'Porte',
     'valor': 'Médio',
     'unidade': None,
     'valor_or

In [27]:
with open("saida.json", "w", encoding="utf-8") as arquivo:
    json.dump(dados_extraidos, arquivo, ensure_ascii=False, indent=4)